# Collapse check: same-cell gene cosine (diagonal vs off-diagonal)

Finished gene-only JEPA **021803 epoch=08** (`gap_vs_copy_src=0.1075`).

The saved gene h5ad is a **dataset-wide mean** per gene, so it cannot test collapse. This notebook forwards **15 random val cells** and **10 present genes each** at **t=1 (90m)**.

For cell $i$ and genes $g,g'$ in that cell:

- **diagonal** $\cos(\hat z_{i,g},\,z^\star_{i,g})$ — did we hit the right gene?
- **off-diagonal** $\cos(\hat z_{i,g},\,z^\star_{i,g'})$ — would a different gene's target look just as good?

Collapse: those two numbers are almost equal and retrieval $\approx 1/10$. Healthy: diagonal clearly higher, retrieval $\gg$ chance.

In [ ]:
import os
import pickle
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_from_disk

REPO = Path("/home/stuke1/perturbgen/Perturbgen")
sys.path.insert(0, str(REPO))

from perturbgen.Dataloaders.datamodule import PerturbGenDataModule
from perturbgen.Model.gene_query_jepa_trainer import (
    GeneQueryJEPATrainer,
    sample_query_batch,
)
from perturbgen.src.utils import read_dataset_files

def resolve_src_dataset_path(tokenized: str) -> str:
    src_dir = Path(tokenized) / "dataset_2000_hvg_src"
    normal = src_dir / "normal.dataset"
    if normal.exists():
        return str(normal)
    cands = sorted(p for p in src_dir.glob("*.dataset"))
    if len(cands) == 1:
        return str(cands[0])
    raise SystemExit(f"cannot pick src dataset in {src_dir}")

os.environ["WANDB_MODE"] = "disabled"

SEED = 0
N_CELLS = 15
N_GENES = 10
TIME_STEP = 1  # 90m
GPU = 0  # 6/7 are the live 30-epoch run

SOD2 = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")
RUN_DIR = (
    SOD2 / "T_perturb" / "res" / "jepa_gene_query_full_atlas"
    / "fzT_encL3_predL3_q128_mixed_lg1_lc0_contr0_vic0_bs16_ep20_lr0.0001_seed0_splitT_poolcls_20260920_021803"
)
CKPT = RUN_DIR / "checkpoints" / (
    "fzT_encL3_predL3_q128_mixed_lg1_lc0_contr0_vic0_bs16_ep20_"
    "lr0.0001_seed0_splitT_poolcls_20260920_021803-epoch=08-gap=0.1075.ckpt"
)
TOKENIZED = Path(
    "/mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps/tokenized_data/LPS_all_tps_2k"
)
SPLIT_PKL = TOKENIZED / "splits" / "stratified_cell_type_harmonized_seed42_90_10.pkl"
MAP_PKL = TOKENIZED / "token_id_to_genename_2000_hvg.pkl"
SYMBOL_PKL = Path("/mnt/sod2-project/csb4/stuke1/Geneformer/geneformer/gene_name_id_dict.pkl")
OUT_CSV = REPO / "docs" / "examples" / "collapse_check_15x10.csv"

assert CKPT.is_file(), CKPT
print("CKPT", CKPT.name)
print("GPU", GPU, "n_cells", N_CELLS, "n_genes", N_GENES, "t", TIME_STEP)

CKPT fzT_encL3_predL3_q128_mixed_lg1_lc0_contr0_vic0_bs16_ep20_lr0.0001_seed0_splitT_poolcls_20260920_021803-epoch=08-gap=0.1075.ckpt
GPU 0 n_cells 15 n_genes 10 t 1


In [2]:
def load_val_indices(path: Path):
    with open(path, "rb") as f:
        split = pickle.load(f)
    return list(split["val_indices"])


def ensembl_to_symbol(path: Path) -> dict:
    with open(path, "rb") as f:
        raw = pickle.load(f)
    out = {}
    for k, v in raw.items():
        ks, vs = str(k), str(v)
        if ks.startswith("ENSG"):
            out[ks] = vs
        elif vs.startswith("ENSG"):
            out[vs] = ks
    return out


val_i = load_val_indices(SPLIT_PKL)
rng = random.Random(SEED)
picked = sorted(rng.sample(val_i, N_CELLS))
print(f"val pool {len(val_i)}; picked {picked}")

src = load_from_disk(resolve_src_dataset_path(str(TOKENIZED)))
tgt = read_dataset_files(f"{TOKENIZED}/dataset_2000_hvg_tgt", "dataset")
dm = PerturbGenDataModule(
    src_dataset=src,
    tgt_datasets=tgt,
    batch_size=N_CELLS,
    num_workers=0,
    shuffle=False,
    split=True,
    pred_tps=[1, 2, 3],
    n_total_tps=3,
    train_indices=picked,
    val_indices=None,
    test_indices=picked,
    var_list=["cell_pairing_index", "time_after_LPS", "cell_type_harmonized"],
    use_weighted_sampler=False,
    seed=SEED,
    strip_tgt_special_tokens=False,
)
dm.setup("test")
batch = next(iter(dm.test_dataloader()))
print({k: tuple(v.shape) if torch.is_tensor(v) else type(v) for k, v in batch.items() if "input_ids" in k or k in ("cell_pairing_index", "cell_type_harmonized")})

val pool 14812; picked [1931, 43020, 43173, 68709, 70383, 73175, 75922, 77237, 78361, 94676, 122928, 126516, 132977, 136067, 146473]
Start datamodule
Define context_tps for testing
{'src_input_ids': (15, 2048), 'tgt_input_ids_t1': (15, 2048), 'tgt_input_ids_t2': (15, 2048), 'tgt_input_ids_t3': (15, 2048)}


In [3]:
device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
print("device", device)
model = GeneQueryJEPATrainer.load_from_checkpoint(
    str(CKPT), map_location="cpu", output_dir=str(RUN_DIR)
)
model.eval()
model.to(device)

batch_dev = {
    k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()
}
# All 10 queries present (no absent slots): 40% shared + 60% tgt-only.
quiz = sample_query_batch(
    src_input_ids_global=batch_dev["src_input_ids"],
    tgt_input_ids_local=batch_dev[f"tgt_input_ids_t{TIME_STEP}"],
    global_to_local=model.global_to_local,
    all_gene_local_ids=model.all_gene_local_ids,
    n_queries=N_GENES,
    frac_shared=0.4,
    frac_tgt_only=0.6,
    rng=random.Random(SEED),
)
gene_ids_local = quiz["gene_ids_local"].to(device)
is_present = quiz["is_present"].to(device)
tgt_position = quiz["tgt_position"].to(device)
print("present per cell", is_present.sum(dim=1).tolist())
assert bool(is_present.all()), "need 10 present genes per cell"

with torch.no_grad():
    out = model.model.forward_one_timestep(
        src_input_ids=model._src_ids_to_encoder_space(batch_dev["src_input_ids"]),
        tgt_input_ids=model._local_ids_to_encoder_space(
            batch_dev[f"tgt_input_ids_t{TIME_STEP}"]
        ),
        query_gene_ids=model._local_ids_to_encoder_space(gene_ids_local),
        query_is_present=is_present,
        query_tgt_position=tgt_position,
        time_step=TIME_STEP,
    )
z_hat = F.normalize(out["z_hat_gene"].float(), dim=-1)  # (B, Q, D)
z_tgt = F.normalize(out["z_tgt_gene"].float(), dim=-1)
cos_ht = torch.einsum("bqd,bkd->bqk", z_hat, z_tgt)  # hat vs tgt
cos_hh = torch.einsum("bqd,bkd->bqk", z_hat, z_hat)  # hat vs hat
print("z_hat", tuple(z_hat.shape))

device cuda:0
present per cell [10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
z_hat (15, 10, 768)


In [4]:
with open(MAP_PKL, "rb") as f:
    id_to_ensembl = pickle.load(f)
sym = ensembl_to_symbol(SYMBOL_PKL)
l2g = model.local_to_global.detach().cpu()

def gene_name(local_id: int) -> str:
    gid = int(l2g[int(local_id)].item()) if int(local_id) < l2g.numel() else int(local_id)
    ens = str(id_to_ensembl.get(gid, id_to_ensembl.get(int(local_id), f"local_{local_id}")))
    return str(sym.get(ens, ens))

eye = torch.eye(N_GENES, device=cos_ht.device, dtype=torch.bool)
rows = []
pair_idx = batch.get(f"cell_pairing_index_t{TIME_STEP}", list(range(N_CELLS)))
ctypes = batch.get(f"cell_type_harmonized_t{TIME_STEP}", ["?"] * N_CELLS)

print(f"{'cell':<6} {'type':<22} {'diag':>7} {'off_ht':>7} {'off_hh':>7} {'retr':>6}  genes")
print("-" * 100)
for i in range(N_CELLS):
    C = cos_ht[i]
    H = cos_hh[i]
    diag = float(C.diag().mean())
    off_ht = float(C[~eye].mean())
    off_hh = float(H[~eye].mean())
    retr = float((C.argmax(dim=-1) == torch.arange(N_GENES, device=C.device)).float().mean())
    names = [gene_name(int(g)) for g in gene_ids_local[i].cpu().tolist()]
    pid = pair_idx[i] if not torch.is_tensor(pair_idx) else int(pair_idx[i])
    ct = ctypes[i] if not torch.is_tensor(ctypes) else str(ctypes[i])
    print(f"{i:<6} {str(ct)[:22]:<22} {diag:7.3f} {off_ht:7.3f} {off_hh:7.3f} {retr:6.2f}  {', '.join(names)}")
    rows.append({
        "cell_i": i,
        "cell_pairing_index": pid,
        "cell_type": str(ct),
        "diag_hat_vs_tgt": diag,
        "off_hat_vs_tgt": off_ht,
        "margin": diag - off_ht,
        "off_hat_vs_hat": off_hh,
        "retrieval_acc": retr,
        "genes": ",".join(names),
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)
print()
print("MEAN  diag={:.3f}  off_hat_tgt={:.3f}  margin={:.3f}  off_hat_hat={:.3f}  retrieval={:.2f}  (chance=0.10)".format(
    df["diag_hat_vs_tgt"].mean(),
    df["off_hat_vs_tgt"].mean(),
    df["margin"].mean(),
    df["off_hat_vs_hat"].mean(),
    df["retrieval_acc"].mean(),
))
print("wrote", OUT_CSV)
df

cell   type                      diag  off_ht  off_hh   retr  genes
----------------------------------------------------------------------------------------------------
0      B cell                   0.879   0.815   0.909   0.80  MLH3, LCN10, CIITA, SPI1, RTN3, MYO16, MFAP3L, POU2AF1, SYK, CALM3
1      CD4+ T cells             0.904   0.821   0.917   0.90  CD3D, DNAJB6, AASS, ARAP2, CYB5R3, SAT1, TLN1, ITM2C, GSTP1, PRDM1
2      CD4+ T cells             0.914   0.842   0.924   0.80  FTH1, HSP90B1, TMSB4X, ANXA1, BATF, MYH9, IQGAP2, TAGAP, KCTD20, C12orf75
3      CD8+ T cells             0.889   0.825   0.924   0.70  TMBIM1, IKZF3, RPL30, CD3E, SPINT2, NCF1, STAT1, KLRC4, PRCD, POU2F2
4      CD8+ T cells             0.881   0.816   0.905   0.50  CD37, PRF1, VIM, KLRG1, MYH9, MTPN, FTH1, FGFBP2, TLN1, NKG7
5      CD8+ T cells             0.887   0.837   0.943   0.60  ACTB, LITAF, CD3E, PTPRCAP, CD8A, VSIG2, HSP90B1, KLRC3, TUBA4A, GCA
6      CD8+ T cells             0.887   0.823   0.92

## Verdict

On these 15 val cells (t=90m, 10 present genes each): mean **diag 0.886** vs **off-diag (hat vs tgt) 0.825**, retrieval **0.68** (chance 0.10).

This is **not full collapse** (wrong gene is a worse match than the right gene). `off_hat_hat` ~0.92 is high because latents sit on a shared cone — that is packing, not “every gene maps to the same vector” (then retrieval would be ~0.10).
